# 05_01_BPE_の_5_手と_4_つのトークナイザでの比

コード：`ch05/blocks/05_01_BPE_の_5_手と_4_つのトークナイザでの比.ipynb`

本文のコードをコードセルで直接実行します。

モデル・データの取得、または長い計算を含みます。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2', 'transformers==5.22.1', 'tiktoken==0.14.0', 'scikit-learn>=1.5,<2', 'pydantic==2.14.2', 'outlines==1.3.3', 'lancedb==0.41.0', 'pyarrow==26.0.1', 'sentence-transformers==6.2.0']], check=True)


In [ ]:
# 5 章：BPE の結合を小さなコーパスで追い、日本語と英語のトークン数を実際のトークナイザで測る
from collections import Counter

# ① BPE：最も多く隣り合う 2 記号を 1 つにまとめる、を繰り返す
corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}      # 単語と出現回数
words = {tuple(w) + ("</w>",): n for w, n in corpus.items()}  # 文字に分け、語末に印
for step in range(1, 6):
    pairs = Counter()
    for sym, n in words.items():
        for a, b in zip(sym, sym[1:]):
            pairs[a, b] += n
    (a, b), cnt = pairs.most_common(1)[0]
    merged = {}
    for sym, n in words.items():
        out, i = [], 0
        while i < len(sym):
            if i + 1 < len(sym) and (sym[i], sym[i + 1]) == (a, b):
                out.append(a + b); i += 2
            else:
                out.append(sym[i]); i += 1
        merged[tuple(out)] = n
    words = merged
    print(f"手 {step}: '{a}' + '{b}' → '{a + b}'（{cnt} 回）  " + "  ".join(" ".join(s) for s in words))

# ② 同じ内容の日本語と英語を、3 つのトークナイザで数える
ja = ("機械学習モデルの性能は、訓練データではなく未知のデータで測る。"
      "訓練データでの成績はいくらでも上げられるが、それは暗記であって汎化ではない。")
en = ("The performance of a machine learning model is measured on unseen data, not on the training data. "
      "Scores on the training data can be raised without limit, but that is memorization, not generalization.")
import tiktoken
from transformers import AutoTokenizer
toks = {"o200k_base（tiktoken）": tiktoken.get_encoding("o200k_base").encode,
        "cl100k_base（tiktoken）": tiktoken.get_encoding("cl100k_base").encode,
        "Qwen2.5": AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct").encode,
        "multilingual-e5": lambda s: AutoTokenizer.from_pretrained("intfloat/multilingual-e5-large")(s, add_special_tokens=False)["input_ids"]}
print(f"日本語 {len(ja)} 文字・英語 {len(en)} 文字（{len(en.split())} 語）")
for name, enc in toks.items():
    nj, ne = len(enc(ja)), len(enc(en))
    print(f"{name:24s} 日本語 {nj:3d}  英語 {ne:3d}  日本語/英語 {nj / ne:.2f}  日本語 1 文字あたり {nj / len(ja):.2f}")
